# Broadcasting: hardware campaigns and convergence

This notebook is the starting point for **hardware acquisition → saved results → PNG figures**. Run all cells with the checked-in settings to review existing data and preview the next experiments. Hardware preparation, submission, collection, and new Monte Carlo computation each have an explicit switch, initially `False`.

1. Select your saved IBM profile, one backend, and new campaign directory names.
2. Review the two campaign budgets and every planned angle.
3. Enable **prepare** to fetch the target and freeze circuits. Inspect timing, mappings, and compiled depths.
4. Enable the desired **submit** cell. Job IDs are saved immediately; kernel restarts do not lose jobs.
5. Enable **collect** after jobs finish, then rerun the analysis and plotting cells.

IBM runs the quantum circuits. **No HPC scripts are needed.** The convergence study runs locally on the CPU. Setup and recovery details are in [README](README.md). All figure exports here are PNG.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "broadcasting").is_dir():
    raise RuntimeError("Open this notebook from the Broadcasting project root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from broadcasting.hardware_campaign import (
    make_campaign_config, plan_campaign, prepare_campaign, load_prepared_campaign,
    submit_campaign, collect_campaign, campaign_status, attach_job,
)
from broadcasting.plotting import plot_hardware_scaling, plot_delay_repeats, save_figure
from scripts.analyze_saved_hardware import load_hardware_runs, analyze
from broadcasting.convergence import (
    load_historical_convergence, plot_convergence, collect_convergence_repeats,
)

plt.rcParams.update({"figure.dpi": 120})
FIGURE_DIR = ROOT / "figures" / "campaigns"
SAVE_PNG = True  # Saves derived plots only; never starts an experiment.


def show_png(fig, filename):
    if SAVE_PNG:
        save_figure(fig, FIGURE_DIR / filename)
    display(fig)
    plt.close(fig)

## Hardware settings

Keep the same explicit backend for both campaigns. The saved profile contains the credentials; no token belongs in this notebook. Change the directory suffix for a new campaign. Reusing the same directory resumes its frozen configuration and existing jobs.

Preparation can be expensive for larger `(M,N)` because resource-state synthesis is generic. The preview reports logical qubits and total shots; the preparation review adds actual compiled depth and mappings. The listed shots are a measurement count, not an estimate of paid device time.

In [ ]:
IBM_PROFILE = "mprest1"
IBM_BACKEND = "EDIT_ME"  # Explicit IBM backend name, e.g. a backend available to this profile.
CAMPAIGN_SEED = 20260908
HARDWARE_REPEATS = 3
SCALING_RUN_DIR = ROOT / "campaigns" / "scaling_01"
DELAY_RUN_DIR = ROOT / "campaigns" / "delay_01"

## 1. Zero-delay scaling: 8,192 shots per case

The default matrix contains **12 cases: M=1,2,3 × N=1,2,3,4**, each repeated three times at **optimization level 3** and zero delay. Within one repeat, cases with the same sender count share the same angles across receiver counts; smaller sender counts use the prefix of the same sampled phase vector. Repeat phases change independently from the PUB shuffle seed. Every actual angle is archived.

Default budget: **3 jobs, 36 circuit evaluations, 294,912 shots**. Edit the explicit lists below to change the matrix.

In [ ]:
SCALING_SENDERS = [1, 2, 3]
SCALING_RECEIVERS = [1, 2, 3, 4]
scaling_config = make_campaign_config(
    "scaling", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    campaign_id="broadcasting-scaling-01", repeats=HARDWARE_REPEATS,
    seed=CAMPAIGN_SEED, sender_counts=SCALING_SENDERS,
    receiver_counts=SCALING_RECEIVERS,
)
assert scaling_config["shots"] == 8192
assert scaling_config["optimization_level"] == 3

## 2. Receiver fidelity versus time: 10,000 shots per delay

Run **M=1,N=2**, with the same delay on both receivers, **three sweeps with different seeded random angles**. Each sweep has 121 delays, `0,50,…,6000 dt`, matching the historical grid. Every point receives **10,000 shots**, at optimization level 3. Each repeat has one angle held constant across its whole sweep.

Default budget: **3 jobs, 363 circuit evaluations, 3,630,000 shots**. Repeats vary both phase and acquisition time, so their spread includes phase dependence and device drift; it is not pure variance at a fixed phase.

`dt` is backend-specific. Preparation reports the corresponding seconds and rejects an unsupported 50-dt step. If that happens, explicitly edit `TAU_VALUES_DT` and use a new directory. **Delays are never rounded or silently rescaled.** For comparisons between different devices, inspect the recorded physical times.

In [ ]:
TAU_VALUES_DT = list(range(0, 6001, 50))
delay_config = make_campaign_config(
    "delay", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    campaign_id="broadcasting-delay-01", repeats=HARDWARE_REPEATS,
    seed=CAMPAIGN_SEED + 1000, tau_values_dt=TAU_VALUES_DT,
)
assert delay_config["shots"] == 10000
assert delay_config["optimization_level"] == 3
campaigns = {
    "scaling": (scaling_config, SCALING_RUN_DIR),
    "delay": (delay_config, DELAY_RUN_DIR),
}

## 3. Preview the complete budget and phases — offline

The full `plans` dictionary includes each submitted position, case, delay, and angle. The compact output below lists every case and each repeat's phase samples. PUBs are interleaved and shuffled deterministically; IBM does not guarantee that submitted order equals chronological execution order.

In [ ]:
plans = {name: plan_campaign(config) for name, (config, _) in campaigns.items()}
for name, plan in plans.items():
    print(f"\n{name.upper()} → {campaigns[name][1]}")
    print(json.dumps({key: value for key, value in plan.items() if key != "repeats"}, indent=2))
    for repeat in plan["repeats"]:
        print(f"Repeat {repeat['repeat_index']} phase seed {repeat['phases']['phase_seed']}:")
        for case, samples in zip(plan["cases"], repeat["phases"]["theta_samples_by_case"]):
            print(f"  {case['id']}: {samples}")
print(f"\nCOMBINED: {sum(plan['jobs'] for plan in plans.values())} jobs; "
      f"{sum(plan['total_shots'] for plan in plans.values()):,} shots")

## 4. Prepare and review — account access, no submission

Enable only the campaign(s) you intend to prepare. This fetches the backend target, checks its timing and qubit capacity, freezes the initial layout for each `(M,N)`, and archives the exact compiled circuits for each distinct phase. Identical phases reuse their compilation; changed phases receive separate archived circuits.

A completed directory is reused only when its configuration matches exactly. If preparation fails, its `plan.json` remains for diagnosis. Fix the error and choose a new directory; an incomplete folder is never mistaken for a completed preparation.

In [ ]:
PREPARE_SCALING = False
PREPARE_DELAY = False

for name, enabled in {"scaling": PREPARE_SCALING, "delay": PREPARE_DELAY}.items():
    if enabled:
        config, run_dir = campaigns[name]
        if (run_dir / "prepared.json").exists():
            load_prepared_campaign(run_dir, expected_config=config)
            print(f"Reusing matching frozen {name} preparation: {run_dir}")
        else:
            prepare_campaign(config, run_dir)
    else:
        print(f"{name}: preparation disabled")

In [ ]:
for name, (config, run_dir) in campaigns.items():
    if (run_dir / "prepared.json").exists():
        bundle = load_prepared_campaign(run_dir, expected_config=config)
        review = bundle["review"]
        print(f"\n{name.upper()}: dt={review['dt_seconds']:.12g} seconds")
        print(f"Delay endpoints: {review['delay_seconds'][0]:.12g} to "
              f"{review['delay_seconds'][-1]:.12g} seconds")
        print(json.dumps({key: value for key, value in review.items()
                          if key not in {"mapping_review", "delay_seconds"}}, indent=2))
        print(json.dumps(campaign_status(run_dir), indent=2))
        # Full per-circuit receiver mappings remain available for inspection:
        # display(bundle["review"]["mapping_review"])
    else:
        print(f"{name}: no completed preparation at {run_dir}")

## 5a. Submit all scaling cases — starts hardware jobs

Set `SUBMIT_SCALING=True` and run this cell after reviewing preparation. It submits the entire `(M,N)` matrix at **8,192 shots per evaluation**, with one job per repeat. Each job ID is written before the next submission. Rerunning this cell skips jobs with receipts.

In [ ]:
SUBMIT_SCALING = False

if SUBMIT_SCALING:
    load_prepared_campaign(SCALING_RUN_DIR, expected_config=scaling_config)
    scaling_job_ids = submit_campaign(SCALING_RUN_DIR)
    print("New scaling jobs:", scaling_job_ids)
else:
    print("Scaling submission disabled")

## 5b. Submit the random-angle delay repeats — starts hardware jobs

Set `SUBMIT_DELAY=True` and run this cell to submit the three sweeps at **10,000 shots per delay**. The phases shown in the preview are the phases in the archived circuits and collected records.

In [ ]:
SUBMIT_DELAY = False

if SUBMIT_DELAY:
    load_prepared_campaign(DELAY_RUN_DIR, expected_config=delay_config)
    delay_job_ids = submit_campaign(DELAY_RUN_DIR)
    print("New delay jobs:", delay_job_ids)
else:
    print("Delay submission disabled")

## 6. Collect or resume — retrieves existing jobs only

Enable collection when jobs finish. This can wait for unfinished jobs. Results are saved independently for each repeat and case as `results/repeat_###_case.json` under the campaign directory. Existing result files are preserved; rerunning resumes missing outputs.

If a connection drops after submission but before a job ID receipt, submission stops as **ambiguous**. Find the job in IBM using the unique tags saved under `attempts/`, then run `attach_job(RUN_DIR, repeat=0, job_id="...")`. It verifies the tags, backend, shots, and exact archived circuit order before linking. Resume collection or submission afterward; do not create a duplicate campaign to retry an ambiguous job.

In [ ]:
COLLECT_SCALING = False
COLLECT_DELAY = False

for name, enabled in {"scaling": COLLECT_SCALING, "delay": COLLECT_DELAY}.items():
    if enabled:
        config, run_dir = campaigns[name]
        load_prepared_campaign(run_dir, expected_config=config)
        print(f"{name}: saved", collect_campaign(run_dir))
        print(json.dumps(campaign_status(run_dir), indent=2))
    else:
        print(f"{name}: collection disabled")

## 7. Existing and collected hardware results — offline PNG plots

The loader includes historical records and campaign `repeat_*.json` recursively, applies documented historical metadata corrections, and removes duplicate records without merging separate cases from one job.

**Scaling:** fidelity on the y-axis, receiver count `N` on the x-axis, colour denotes sender count `M`. Only optimization-level-3 records at zero delay are used. Markers show the receiver mean and vertical bars the receiver minimum–maximum; these bars are not statistical confidence intervals. Separate jobs/angles remain separate points. Historical device/date/shot differences remain in the plotted-point metadata.

**Delay:** each receiver's fidelity is plotted against actual recorded time, with separate panels for each repeat/angle. The default display shows the latest two historical two-receiver sweeps plus every newly collected delay repeat.

In [ ]:
hardware_runs = load_hardware_runs(ROOT / "results", ROOT / "campaigns")
print(f"Loaded {len(hardware_runs)} distinct hardware records")

In [ ]:
scaling_figure = plot_hardware_scaling(hardware_runs)
show_png(scaling_figure, "hardware_scaling_opt3.png")
# Exact plotted jobs, settings, angles, means and receiver ranges:
scaling_points = scaling_figure.broadcasting_points
print(f"Scaling plot contains {len(scaling_points)} distinct job/case/angle points")

In [ ]:
eligible_delay = [run for run in hardware_runs
                  if run.get("optimization_level") == 3 and run.get("M") == 1 and run.get("N") == 2
                  and run.get("sweep", {}).get("axis") == "tau"
                  and len(run["sweep"]["values"]) > 1]
new_delay = [run for run in eligible_delay
             if (run.get("metadata") or {}).get("campaign", {}).get("campaign_id") == delay_config["campaign_id"]]
historical_delay = [run for run in eligible_delay if not (run.get("metadata") or {}).get("campaign")]
historical_delay = sorted(historical_delay, key=lambda run: run.get("timestamp") or "")[-2:]
displayed_delay_runs = historical_delay + new_delay
if displayed_delay_runs:
    delay_figure = plot_delay_repeats(displayed_delay_runs, max_columns=2)
    show_png(delay_figure, "receiver_fidelity_vs_time_repeats.png")
else:
    print("No eligible delay sweeps found. Rerun after collection.")

## 8. Write the detailed statistics report — offline, optional

The plots above are available without writing new analysis JSON. Enable the cell below after collection for per-case/per-angle/per-delay receiver, joint and worst-receiver statistics, receiver differences, covariance, and exploratory periodicity. It writes one separate report for each selected campaign. Publication selection remains explicit: update the pinned figure sources and manuscript captions after reviewing the new data.

In [ ]:
WRITE_ANALYSIS = False

if WRITE_ANALYSIS:
    for name, (_, run_dir) in campaigns.items():
        if list((run_dir / "results").glob("repeat_*.json")):
            output_dir = ROOT / "analysis" / "campaigns" / run_dir.name
            analyze(output_dir, results_dir=run_dir / "results")
            print(f"{name} analysis: {output_dir / 'report.md'}")
        else:
            print(f"{name}: no collected records yet")

## 9. Restored Monte Carlo-to-exact convergence — existing data

The log–log plot uses the original **M=1,N=2**, `alpha=1/√2`, `theta=[0]`, QEC enabled, sender outcome `[0]`, and **21 equally spaced depolarizing probabilities from 0 to 1**. The trajectory counts are **50,100,200,500,1000,2000,5000,10000,50000,100000**.

The historical seed-0 receiver errors were recovered from saved notebook summaries. They are labelled as historical summary data because full raw fidelity arrays for those points were not archived. New repetitions save full exact and sampled arrays and are overlaid separately. This figure can be displayed immediately; no simulation is needed to restore it.

In [ ]:
CONVERGENCE_DIR = ROOT / "results" / "convergence"
convergence_study = load_historical_convergence(archive_dir=CONVERGENCE_DIR)
print("Historical comparison configuration:", convergence_study.config)
print("Trajectory counts:", convergence_study.sample_counts)
convergence_figure = plot_convergence(convergence_study)
show_png(convergence_figure, "mc_sampling_convergence.png")

## 10. Add two matching independent convergence repetitions — local CPU

Enable `RUN_CONVERGENCE` to run **the same full parameter grid** again with seeds **1 and 2**. This is a local Monte Carlo calculation, not an IBM job, and the high-count points can take substantial time. At 100,000 trajectories the stored encoded vectors alone use about 4.9 GB, with additional working buffers; use a machine with sufficient free memory. Every completed sweep is checkpointed; rerunning the same batch directory resumes it. Keep the fixed batch directory to recover interrupted work. Use a new directory and new seeds for additional independent repetitions.

After it finishes, this cell reloads historical and new data and refreshes the same PNG overlay. The `n⁻¹ᐟ²` line is a reference scaling law; descriptive fitted slopes are not confidence intervals.

In [ ]:
RUN_CONVERGENCE = False
CONVERGENCE_SEEDS = [1, 2]
CONVERGENCE_BATCH_DIR = CONVERGENCE_DIR / "repeats_01"

if RUN_CONVERGENCE:
    collect_convergence_repeats(
        convergence_study, repeats=2, seeds=CONVERGENCE_SEEDS,
        batch_dir=CONVERGENCE_BATCH_DIR,
    )
    convergence_study = load_historical_convergence(archive_dir=CONVERGENCE_DIR)
    convergence_figure = plot_convergence(convergence_study)
    show_png(convergence_figure, "mc_sampling_convergence.png")
else:
    print("New convergence simulation disabled; historical and already-saved repetitions are displayed above.")

## Figures and manuscript handoff

PNG outputs are written under `figures/campaigns/`: `hardware_scaling_opt3.png`, `receiver_fidelity_vs_time_repeats.png`, and `mc_sampling_convergence.png`. The first two provide candidates for the manuscript's hardware scaling and delay figures. The convergence plot restores the simulation scaling analysis and includes new repetitions once collected.

Keep each campaign folder, including preparation, attempts, receipts, and raw results. The manuscript figure generator uses a pinned source manifest: review new data, choose the final sources, and update captions/settings before replacing publication figures. The notebook does not silently replace that selection. [README](README.md) remains the single project reference.